In [10]:
import pandas as pd

users_path = './dataset/PP_users.csv'
interactions_path = './dataset/RAW_interactions.csv'
recipes_path = './dataset/RAW_recipes.csv'

df_users = pd.read_csv(users_path)
df_interactions = pd.read_csv(interactions_path)
df_recipes = pd.read_csv(recipes_path)


PROFIL DES UTILISATEURS CONTRIBUANTS LE PLUS AU SITE :

In [6]:
print(df_users.columns)
print(df_users.head())
# Compter le nombre d'utilisateurs
nombre_utilisateurs = df_users['u'].nunique()
print(f"Nombre total d'utilisateurs : {nombre_utilisateurs}")


Index(['u', 'techniques', 'items', 'n_items', 'ratings', 'n_ratings'], dtype='object')
   u                                         techniques  \
0  0  [8, 0, 0, 5, 6, 0, 0, 1, 0, 9, 1, 0, 0, 0, 1, ...   
1  1  [11, 0, 0, 2, 12, 0, 0, 0, 0, 14, 5, 0, 0, 0, ...   
2  2  [13, 0, 0, 7, 5, 0, 1, 2, 1, 11, 0, 1, 0, 0, 1...   
3  3  [498, 13, 4, 218, 376, 3, 2, 33, 16, 591, 10, ...   
4  4  [161, 1, 1, 86, 93, 0, 0, 11, 2, 141, 0, 16, 0...   

                                               items  n_items  \
0  [1118, 27680, 32541, 137353, 16428, 28815, 658...       31   
1  [122140, 77036, 156817, 76957, 68818, 155600, ...       39   
2  [168054, 87218, 35731, 1, 20475, 9039, 124834,...       27   
3  [163193, 156352, 102888, 19914, 169438, 55772,...     1513   
4  [72857, 38652, 160427, 55772, 119999, 141777, ...      376   

                                             ratings  n_ratings  
0  [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 4.0, 4.0, ...         31  
1  [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5

In [4]:
print(df_interactions.columns)

print(df_interactions.head())

Index(['user_id', 'recipe_id', 'date', 'rating', 'review'], dtype='object')


In [11]:
print(df_recipes.columns)

print(df_recipes.head())

Index(['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'ingredients',
       'n_ingredients'],
      dtype='object')
                                         name      id  minutes  \
0  arriba   baked winter squash mexican style  137739       55   
1            a bit different  breakfast pizza   31490       30   
2                   all in the kitchen  chili  112140      130   
3                          alouette  potatoes   59389       45   
4          amish  tomato ketchup  for canning   44061      190   

   contributor_id   submitted  \
0           47892  2005-09-16   
1           26278  2002-06-17   
2          196586  2005-02-25   
3           68585  2003-04-14   
4           41706  2002-10-25   

                                                tags  \
0  ['60-minutes-or-less', 'time-to-make', 'course...   
1  ['30-minutes-or-less', 'time-to-make', 'course...   
2  ['time-to-make', 'course', 'preparation', 'ma

In [16]:
# Profil des utilisateurs contribuant le plus au site avec la moyenne de note
# Joindre les données des utilisateurs avec les interactions sur user_id
df_contributions = df_users.merge(df_interactions, left_on='u', right_on='user_id', how='inner')

# Grouper par utilisateur pour obtenir le nombre de recettes, nombre de notes et la moyenne des notes
profil_utilisateurs = df_contributions.groupby('user_id').agg({
    'recipe_id': 'nunique',  # Nombre unique de recettes
    'rating': ['count', 'mean']  # Nombre de notes données et moyenne des notes
}).reset_index()

# Aplatir les colonnes multi-niveaux
profil_utilisateurs.columns = ['user_id', 'n_recettes_contribuees', 'n_notes_donnees', 'moyenne_notes']

# Trier par les utilisateurs ayant le plus contribué
top_contributeurs = profil_utilisateurs.sort_values(by='n_recettes_contribuees', ascending=False).head(10)

# Afficher les top contributeurs avec leur moyenne de note
print(top_contributeurs)


      user_id  n_recettes_contribuees  n_notes_donnees  moyenne_notes
193      4470                    2739             2739       4.785323
234      5060                    1741             1741       4.686387
1316    17803                    1515             1515       4.509571
562      8688                    1492             1492       4.901475
377      6357                    1455             1455       4.526460
653      9869                    1161             1161       4.841516
953     13483                    1076             1076       4.587361
1741    21752                     905              905       4.710497
1762    22015                     852              852       4.690141
555      8629                     804              804       4.298507


CARACTERISTIQUE DE LA RECETTE LA PLUS POPULAIRE

In [12]:
# Analyzing Popular Recipes
# Merge interactions with recipes on recipe_id
df_popular_recipes = df_interactions.merge(df_recipes, left_on='recipe_id', right_on='id')

# Group by recipe and calculate the average rating
popular_recipes = df_popular_recipes.groupby('name').agg({
    'rating': 'mean',  # Average rating
    'minutes': 'mean',  # Average time to prepare
    'n_ingredients': 'mean',  # Average number of ingredients
    'n_steps': 'mean'   # Average number of steps
}).reset_index()

# Sort by the highest average rating
top_rated_recipes = popular_recipes.sort_values(by='rating', ascending=False).head(10)
print(top_rated_recipes)


                                                     name  rating  minutes  \
230184                          zydeco ya ya deviled eggs     5.0     40.0   
109410                                  idaho potato cake     5.0     60.0   
109459                          ila s mustard cauliflower     5.0      7.0   
109458                                 ila s iced tequila     5.0     15.0   
109456                                    ila s fruit dip     5.0      5.0   
109455                              ila s freckled pecans     5.0     25.0   
109452                                     ila s coleslaw     5.0     15.0   
109447                                  ila s apple crisp     5.0     45.0   
109446               ila  bert and dylan s cashew chicken     5.0     90.0   
109445  il linguine   la salsa di pesto con frutta del...     5.0     45.0   

        n_ingredients  n_steps  
230184            8.0      7.0  
109410           10.0      7.0  
109459            3.0      2.0  
109458   

In [13]:
# Recipe Recommendation Based on Available Time
def recommend_recipes_by_time(time_available):
    # Filter recipes by cooking time
    recommended_recipes = df_recipes[df_recipes['minutes'] <= time_available]
    
    # Sort by rating (if available)
    df_popular_recipes = df_interactions.merge(recommended_recipes, left_on='recipe_id', right_on='id')
    top_recommendations = df_popular_recipes.groupby('name').agg({
        'rating': 'mean'
    }).reset_index().sort_values(by='rating', ascending=False).head(10)
    
    return top_recommendations

# Example usage: Recommend recipes that can be made in under 30 minutes
recommend_recipes_by_time(30)


,name,rating
0,007 martini,5.0
53732,luscious berries with custard sauce,5.0
53702,lump crabmeat and avocado crostini,5.0
53704,lumumba swiss hot chocolate with peppercorns,5.0
53705,luna s red chile and pork over stuffing,5.0
53708,lunch box fun due,5.0
53709,lunch box hot hot dogs,5.0
53715,lunch lady s irresistible asian salad dressing,5.0
53716,lunch meat sandwich menu lite bleu,5.0
53718,lunchable cracker stacker lunch,5.0


In [14]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Extract relevant recipe features (number of steps, number of ingredients, etc.)
recipe_features = df_recipes[['n_steps', 'n_ingredients', 'minutes']]

# Apply PCA for dimensionality reduction
pca = PCA(n_components=2)
recipe_pca = pca.fit_transform(recipe_features)

# Create a scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(recipe_pca[:, 0], recipe_pca[:, 1], alpha=0.5)
plt.title('Recipe Clusters based on Steps, Ingredients, and Cooking Time')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.show()


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
import random

# Generate a recipe name based on ingredients and techniques
def generate_recipe_name(recipe_id):
    # Get the ingredients and techniques for a specific recipe
    recipe = df_recipes[df_recipes['id'] == recipe_id].iloc[0]
    ingredients = recipe['ingredients'].split(",")[:3]  # Take the first 3 ingredients
    name = f"{' '.join(random.sample(ingredients, len(ingredients)))} Delight"
    return name

# Example: Generate a name for a specific recipe
print(generate_recipe_name(12345))  # Replace with a valid recipe_id


In [ ]:
# Merge recipe data with interactions to get ratings
df_recipe_ratings = df_recipes.merge(df_interactions, left_on='id', right_on='recipe_id')

# Convert 'submitted' date to datetime format
df_recipe_ratings['submitted'] = pd.to_datetime(df_recipe_ratings['submitted'])

# Calculate how long each user has been active (in days)
df_recipe_ratings['days_active'] = (pd.to_datetime('today') - df_recipe_ratings['submitted']).dt.days

# Group by user and calculate their average rating and the length of time they've been active
user_ratings = df_recipe_ratings.groupby('contributor_id').agg({
    'rating': 'mean',
    'days_active': 'mean'
}).reset_index()

# Check the correlation between activity length and rating
correlation = user_ratings['days_active'].corr(user_ratings['rating'])
print(f"Correlation between length of activity and rating: {correlation}")
